# 01b — Source verification: are we reading the data correctly?

**Canvas project group:** 37 · **Members:** Nat Toms, Owen Cruz-Abrams, Martha le Huray

Notebook 01 explored *what* the sources contain. This notebook checks that **our pipeline reads them correctly**,
using independent cross-checks:

1. **SNOTEL parser vs an independent NRCS endpoint.** We compare our AWDB REST data (parsed by `src/parse.py`) against
   the NRCS *Report Generator* CSV for the same stations, value by value.
2. **SNOTEL date semantics.** We pin down what the date on an AWDB value means (`periodRef=START` vs `END`).
3. **Open-Meteo daily aggregation and timezone.** We check that the daily values equal the local-day aggregate of the hourly values.
4. **Unit tests** for the lineage, parsing and integrity code (`tests/`).

Lineage: Report Generator CSVs are downloaded through `src/lineage_text.py::fetch_text`. It follows the same contract as
`lineage.fetch`: raw bytes stored unmodified, a MANIFEST row, an honest User-Agent, and offline-mode support.

In [1]:
import sys, io, os, subprocess
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
import lineage, lineage_text, parse, sites
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 30)

STATIONS = {"547:CO:SNTL": "Ivanhoe (Aspen area)", "784:CA:SNTL": "Palisades Tahoe"}
RG_ELEMENTS = ["WTEQ", "SNWD", "TAVG", "TMAX", "TMIN", "PREC", "PRCP"]

def awdb_daily(trip):
    # Exactly the request notebook 01 made (same cache file), so we verify the data we actually use.
    p = {"stationTriplets": trip, "elements": ",".join(sites.SNOTEL_ELEMENTS), "duration": "DAILY",
         "beginDate": sites.SNOTEL_PERIOD[0], "endDate": sites.SNOTEL_PERIOD[1], "periodRef": "END",
         "returnFlags": "true", "returnOriginalValues": "true", "returnSuspectData": "true"}
    return parse.snotel_to_long(lineage.snotel("data", p, name=f"daily_{trip.replace(':', '_')}"))

def report_generator(trip):
    txt = lineage_text.fetch_text(lineage_text.report_generator_url(trip, RG_ELEMENTS),
                                  source="snotel_report_generator", name=f"rg_daily_{trip.replace(':', '_')}")
    header = [l for l in txt.splitlines() if l.startswith("#")]
    df = pd.read_csv(io.StringIO(txt), comment="#")
    return df, header

## 1. AWDB REST (our pipeline) vs NRCS Report Generator CSV
The Report Generator is NRCS's own human-facing download tool. It is a separate endpoint and a separate format (CSV),
so agreement shows that our request parameters and `parse.snotel_to_long` reproduce NRCS's published values.
First, what the CSV says about its own columns:

In [2]:
rg = {}
for trip, label in STATIONS.items():
    df, header = report_generator(trip)
    rg[trip] = df
    print(f"== {label}: {df.shape[0]} rows, {df.Date.min()} -> {df.Date.max()}")
    print("   columns:", list(df.columns))
print()
print("\n".join(l for l in header if any(k in l for k in ["Report Instantaneous", "Instantaneous - Start of Day", "Day   ", "temperature data contains", "subject to revision"])))

== Ivanhoe (Aspen area): 12824 rows, 1991-10-01 -> 2026-11-09
   columns: ['Date', 'Snow Water Equivalent (in) Start of Day Values', 'Snow Depth (in) Start of Day Values', 'Air Temperature Average (degF)', 'Air Temperature Maximum (degF)', 'Air Temperature Minimum (degF)', 'Precipitation Accumulation (in) Start of Day Values', 'Precipitation Increment (in)']
== Palisades Tahoe: 16841 rows, 1980-10-01 -> 2026-11-09
   columns: ['Date', 'Snow Water Equivalent (in) Start of Day Values', 'Snow Depth (in) Start of Day Values', 'Air Temperature Average (degF)', 'Air Temperature Maximum (degF)', 'Air Temperature Minimum (degF)', 'Precipitation Accumulation (in) Start of Day Values', 'Precipitation Increment (in)']

# database are subject to revision regardless of indicated Quality Assurance level. 
# SNOTEL air temperature data contains a known bias. This bias is rooted in the sensor 
# Report Instantaneous Data As: Start of Day
# Snow Water Equivalent       Value       None           Instant

Note the header: SWE, snow depth and accumulated precipitation are **instantaneous "Start of Day" readings**, while
temperatures and the precipitation increment are **daily aggregates**. Below we compare every value, also trying
a one-day shift of the AWDB dates.

In [3]:
rows, aligned = [], {}
for trip, label in STATIONS.items():
    r = rg[trip].copy()
    r.columns = ["date"] + RG_ELEMENTS
    r["date"] = pd.to_datetime(r["date"]); r = r.set_index("date")
    aw = awdb_daily(trip)
    for el in RG_ELEMENTS:
        a = aw[aw.element == el].set_index("date")
        for shift in (-1, 0, 1):
            s = a["value"].copy(); s.index = s.index + pd.Timedelta(days=shift)  # AWDB date d compared with RG date d+shift
            j = pd.concat([s.rename("awdb"), r[el].rename("rg")], axis=1, join="inner")
            both = j.dropna()
            rows.append({"station": label, "element": el, "AWDB date +": shift, "n both present": len(both),
                         "% exact match": round(100 * (both.awdb == both.rg).mean(), 2),
                         "AWDB only": int((j.awdb.notna() & j.rg.isna()).sum()),
                         "RG only": int((j.rg.notna() & j.awdb.isna()).sum())})
        aligned[(trip, el)] = (a, r[el])
match = pd.DataFrame(rows)
match.pivot_table(index=["station", "element"], columns="AWDB date +", values="% exact match")

AWDB date +                      -1       0       1
station              element                       
Ivanhoe (Aspen area) PRCP     48.71  100.00   48.71
                     PREC     42.35   60.43   99.73
                     SNWD     49.73   56.10  100.00
                     TAVG      1.67  100.00    1.67
                     TMAX      1.24  100.00    1.24
                     TMIN      2.05  100.00    2.05
                     WTEQ     53.94   64.51  100.00
Palisades Tahoe      PRCP     63.52  100.00   63.52
                     PREC     61.01   72.25   99.73
                     SNWD     48.77   54.19  100.00
                     TAVG      4.11  100.00    4.11
                     TMAX      3.72  100.00    3.72
                     TMIN      4.44  100.00    4.44
                     WTEQ     53.40   61.57  100.00

In [4]:
best = match.loc[match.groupby(["station", "element"])["% exact match"].idxmax()].set_index(["station", "element"])
best

AWDB date +  n both present  % exact match  AWDB only  RG only
station              element                                                                
Ivanhoe (Aspen area) PRCP               0           12723         100.00          0        0
                     PREC               1           12723          99.73          0        0
                     SNWD               1           10202         100.00          0        0
                     TAVG               0           12181         100.00          5        0
                     TMAX               0           12186         100.00          0        0
                     TMIN               0           12145         100.00         38        0
                     WTEQ               1           12724         100.00          0        0
Palisades Tahoe      PRCP               0           16740         100.00          0        0
                     PREC               1           16741          99.73          0        0
                     SNWD               1            8748         100.00          0        0
                     TAVG               0           13343         100.00          1        0
                     TMAX               0           13360         100.00          1        0
                     TMIN               0           13340         100.00          5        0
                     WTEQ               1           16741         100.00          0        0

**Result.** Every element matches 100% (or ~99.7% for PREC) at exactly one alignment:
- **Daily aggregates (TAVG, TMAX, TMIN, PRCP): same date.**
- **Instantaneous readings (SNWD, WTEQ, PREC): AWDB date + 1 day = Report Generator date.** With `periodRef=END`, the
  AWDB value dated *d* is the reading at the **end of day *d*** (midnight), which the Report Generator calls the
  start of day *d*+1. Same physical reading, different label.

Next we explain every remaining mismatch.

In [5]:
resid = []
for trip, label in STATIONS.items():
    for el in RG_ELEMENTS:
        a, r_el = aligned[(trip, el)]
        shift = 1 if el in ("SNWD", "WTEQ", "PREC") else 0
        a = a.copy(); a.index = a.index + pd.Timedelta(days=shift)
        j = a.join(r_el.rename("rg"), how="inner")
        mm = j[~((j["value"] == j["rg"]) | (j["value"].isna() & j["rg"].isna()))].copy()
        if len(mm):
            mm["station"], mm["element"] = label, el
            resid.append(mm.reset_index())
resid = pd.concat(resid, ignore_index=True)
resid["kind"] = np.select(
    [resid.element.eq("PREC") & resid.date.dt.strftime("%m-%d").eq("10-01") & resid.rg.eq(0),
     resid.qc_flag.eq("S") & resid.rg.isna()],
    ["PREC water-year reset (RG shows 0 at start of Oct 1)", "suspect (S) value: AWDB returns it (returnSuspectData=true), RG hides it"],
    default="UNEXPLAINED")
display(resid.groupby(["station", "element", "kind"]).size().to_frame("n"))
print("Examples:")
display(resid.groupby("kind").head(3)[["station", "element", "date", "value", "qc_flag", "orig_value", "rg", "kind"]])
print("Unexplained mismatches:", int(resid.kind.eq("UNEXPLAINED").sum()))

n
station              element kind                                                  
Ivanhoe (Aspen area) PREC    PREC water-year reset (RG shows 0 at start of O...  34
                     TAVG    suspect (S) value: AWDB returns it (returnSuspe...   5
                     TMIN    suspect (S) value: AWDB returns it (returnSuspe...  38
Palisades Tahoe      PREC    PREC water-year reset (RG shows 0 at start of O...  46
                     TAVG    suspect (S) value: AWDB returns it (returnSuspe...   1
                     TMAX    suspect (S) value: AWDB returns it (returnSuspe...   1
                     TMIN    suspect (S) value: AWDB returns it (returnSuspe...   5

Examples:


,station,element,date,value,qc_flag,orig_value,rg,kind
0,Ivanhoe (Aspen area),TAVG,1994-06-22,44.6,S,44.6,NaN,suspect (S) value: AWDB returns it (returnSusp...
1,Ivanhoe (Aspen area),TAVG,1994-06-23,53.8,S,53.8,NaN,suspect (S) value: AWDB returns it (returnSusp...
2,Ivanhoe (Aspen area),TAVG,2009-04-05,-11.2,S,-11.2,NaN,suspect (S) value: AWDB returns it (returnSusp...
43,Ivanhoe (Aspen area),PREC,1992-10-01,31.8,E,31.8,0.0,PREC water-year reset (RG shows 0 at start of ...
44,Ivanhoe (Aspen area),PREC,1993-10-01,40.5,E,40.5,0.0,PREC water-year reset (RG shows 0 at start of ...
45,Ivanhoe (Aspen area),PREC,1994-10-01,27.6,V,27.6,0.0,PREC water-year reset (RG shows 0 at start of ...


Unexplained mismatches: 0


**Every residual mismatch is explained:**
- **PREC on Oct 1:** the water year starts Oct 1, and the Report Generator shows the reset value (0) at the *start*
  of Oct 1, while the end-of-day Sep 30 reading is the season total. This is consistent with the end-of-day semantics above.
  *Cleaning implication:* season-accumulated PREC resets every Oct 1, so it must be differenced within a water year.
- **Suspect values:** the extra AWDB values are all QC flag **S**. They appear only because we request
  `returnSuspectData=true` (D-009); NRCS's own report hides them. Some are absurd, such as TMIN ≈ 2148 °F, which confirms that suspect values
  must be removed in cleaning.

## 2. What does the date on an AWDB value mean? `periodRef=START` vs `END`
A small direct test: Ivanhoe, first 10 days of January 2020, requested both ways.

In [6]:
pr = {}
for ref in ["START", "END"]:
    p = {"stationTriplets": "547:CO:SNTL", "elements": "SNWD,TAVG", "duration": "DAILY",
         "beginDate": "2020-01-01", "endDate": "2020-01-10", "periodRef": ref, "returnFlags": "true"}
    d = parse.snotel_to_long(lineage.snotel("data", p, name=f"periodref_{ref}_547_CO_SNTL_2020-01"))
    pr[ref] = d.pivot(index="date", columns="element", values="value")
pr = pd.concat(pr, axis=1)
display(pr)
print("SNWD END(d) == START(d+1):", bool((pr[("END", "SNWD")].iloc[:-1].values == pr[("START", "SNWD")].iloc[1:].values).all()))
print("TAVG identical under START and END:", bool((pr[("END", "TAVG")] == pr[("START", "TAVG")]).all()))

START         END      
element     SNWD  TAVG  SNWD  TAVG
date                              
2020-01-01  34.0  15.3  40.0  15.3
2020-01-02  40.0  12.0  44.0  12.0
2020-01-03  44.0  10.9  41.0  10.9
2020-01-04  41.0  18.7  40.0  18.7
2020-01-05  40.0  16.3  39.0  16.3
2020-01-06  39.0   8.6  38.0   8.6
2020-01-07  38.0  16.5  38.0  16.5
2020-01-08  38.0  19.8  37.0  19.8
2020-01-09  37.0  12.2  38.0  12.2
2020-01-10  38.0   5.0  38.0   5.0

SNWD END(d) == START(d+1): True
TAVG identical under START and END: True


**Result → D-009 confirmed.** With `periodRef=END`, **SNWD on date *d* is the snow depth at the end of day *d***,
after that day's snowfall and melt. Daily aggregates (TAVG) are the same either way. In practice:
- a feature row for day *d* can pair Open-Meteo's weather for day *d* with SNOTEL depth at the end of *d*;
- the prediction target "snow depth 30 days after *t*" = SNWD dated *t*+30 (end of that day).

## 3. Open-Meteo: are the daily values the local-day aggregate of the hourly values?
Notebook 01 and the modelling use Open-Meteo **daily** variables with `timezone=auto`. We take responses where hourly and
daily values came from the **same** request, re-aggregate the hourly values by **local** calendar day, and compare them with the daily
values. As a contrast, we also aggregate by the **UTC** day.

**3a (snow depth, offline).** This reuses notebook 01 §1.4's cached request (Aspen Mountain, winter 2023–24, `timezone=auto`), which
contains daily `snow_depth_mean/max` and hourly `snow_depth`. We locate it through MANIFEST.csv and check its SHA-256.

In [7]:
import hashlib, json
man = lineage.manifest()
agg_rows = []
for model in ["era5_land", "era5_seamless"]:
    row = man[(man.name == f"vars_aspen_{model}") & (man.http_status == 200)].iloc[-1]
    raw = (ROOT / row.file).read_bytes()
    assert hashlib.sha256(raw).hexdigest() == row.sha256, "raw file changed since download!"
    j = json.loads(raw)
    h = pd.DataFrame(j["hourly"])[["time", "snow_depth"]]; h["time"] = pd.to_datetime(h["time"])
    local = h.groupby(h.time.dt.normalize())["snow_depth"].agg(["mean", "max", "size"])
    utc = h.assign(u=(h.time - pd.Timedelta(seconds=j["utc_offset_seconds"])).dt.normalize()).groupby("u")["snow_depth"].agg(["mean", "max"])
    d = parse.open_meteo_daily(j)
    for col, agg in [("snow_depth_mean", "mean"), ("snow_depth_max", "max")]:
        for frame, label in [(local, "local day"), (utc, "UTC day (contrast)")]:
            diff = (d[col] - frame[agg]).abs().dropna()
            agg_rows.append({"model": model, "timezone": j["timezone"], "daily variable": col, "hourly aggregated by": label,
                             "days": len(diff), "max |diff| (m)": round(diff.max(), 4),
                             "% days within rounding (0.005 m)": round(100 * (diff <= 0.005 + 1e-9).mean(), 1)})
    print(model, row.file, "| hours per local day:", local["size"].value_counts().to_dict())
agg_tab = pd.DataFrame(agg_rows)
agg_tab

era5_land data/raw/open_meteo/vars_aspen_era5_land__64a63c07ba.json | hours per local day: {24: 182}
era5_seamless data/raw/open_meteo/vars_aspen_era5_seamless__1b20bc0f5c.json | hours per local day: {24: 182}


,model,timezone,daily variable,hourly aggregated by,days,max |diff| (m),% days within rounding (0.005 m)
0,era5_land,America/Denver,snow_depth_mean,local day,182,0.0050,100.0
1,era5_land,America/Denver,snow_depth_mean,UTC day (contrast),182,0.0417,50.5
2,era5_land,America/Denver,snow_depth_max,local day,182,0.0000,100.0
3,era5_land,America/Denver,snow_depth_max,UTC day (contrast),182,0.0800,66.5
4,era5_seamless,America/Denver,snow_depth_mean,local day,182,0.0050,100.0
5,era5_seamless,America/Denver,snow_depth_mean,UTC day (contrast),182,0.0417,50.5
6,era5_seamless,America/Denver,snow_depth_max,local day,182,0.0000,100.0
7,era5_seamless,America/Denver,snow_depth_max,UTC day (contrast),182,0.0800,66.5


**3b (temperature, station-level).** The same check for `temperature_2m_mean` at two SNOTEL stations (station coordinates and elevation,
winter 2022–23). Notebook 01 made many requests, so Open-Meteo's free tier may still be rate-limiting. If so, this cell says so
instead of failing. **Re-run it later** and the result will be filled in from the cache.

In [8]:
meta = pd.DataFrame(lineage.snotel("stations", {"stationTriplets": ",".join(sites.CANDIDATE_STATIONS),
                                                "returnStationElements": "true"}, name="stations_candidates_elements")).set_index("stationTriplet")
t_rows = []
for t in ["542:CO:SNTL", "784:CA:SNTL"]:
    r = meta.loc[t]; elev = round(r.elevation * 0.3048)
    p = dict(latitude=r.latitude, longitude=r.longitude, elevation=float(elev), start_date="2022-11-01", end_date="2023-04-30",
             models="era5_land", timezone="auto", hourly=["snow_depth", "temperature_2m"],
             daily=["snow_depth_mean", "temperature_2m_mean", "snow_depth_max"])
    try:
        j = lineage.open_meteo(p, name=f"aggcheck_{t.replace(':', '_')}", retries=1)
    except (RuntimeError, FileNotFoundError) as e:
        print(f"{r['name']}: NOT VERIFIED YET ({str(e)[:90]}...). Re-run this cell later.")
        continue
    h = pd.DataFrame(j["hourly"]); h["time"] = pd.to_datetime(h["time"])
    local = h.groupby(h.time.dt.normalize())["temperature_2m"].mean()
    utc = h.assign(u=(h.time - pd.Timedelta(seconds=j["utc_offset_seconds"])).dt.normalize()).groupby("u")["temperature_2m"].mean()
    d = parse.open_meteo_daily(j)
    for frame, label in [(local, "local day"), (utc, "UTC day (contrast)")]:
        diff = (d["temperature_2m_mean"] - frame).abs().dropna()
        t_rows.append({"station": r["name"], "timezone": j["timezone"], "hourly aggregated by": label, "days": len(diff),
                       "max |diff| (°C)": round(diff.max(), 3), "% days within rounding (0.05 °C)": round(100 * (diff <= 0.05 + 1e-9).mean(), 1)})
pd.DataFrame(t_rows) if t_rows else "temperature aggregation check pending (rate limit)"

Independence Pass: NOT VERIFIED YET (HTTP 429 for https://archive-api.open-meteo.com/v1/archive?daily=snow_depth_mean,temperatu...). Re-run this cell later.


Palisades Tahoe: NOT VERIFIED YET (HTTP 429 for https://archive-api.open-meteo.com/v1/archive?daily=snow_depth_mean,temperatu...). Re-run this cell later.


'temperature aggregation check pending (rate limit)'

## 4. Unit tests
`tests/test_lineage_parse.py` covers:
- canonical URLs;
- caching, with raw bytes stored unmodified;
- manifest rows (sha256, status, User-Agent);
- 400s cached as evidence, 429/5xx logged but not cached, and retries;
- offline mode;
- manifest migration (legacy User-Agent back-fill);
- `lineage_text`, `verify_raw` (changed, missing and orphan files), `winter_of` edge cases, and the parsers.

The tests use no network; HTTP is mocked.

In [9]:
res = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider", str(ROOT / "tests" / "test_lineage_parse.py")],
                     capture_output=True, text=True, cwd=ROOT)
print(res.stdout[-1500:] or res.stderr[-1500:])

........................                                                 [100%]
24 passed in 0.23s



In [10]:
res = subprocess.run([sys.executable, str(ROOT / "src" / "verify_raw.py")], capture_output=True, text=True, cwd=ROOT / "src")
print(res.stdout)

85 cached files checked, 0 problem(s)




## 5. Summary of verification results

| Check | Result | Consequence |
|---|---|---|
| AWDB parser vs NRCS Report Generator (Ivanhoe, Palisades Tahoe; 7 elements; ~12.7k and ~16.7k days) | **100% exact match** for WTEQ, SNWD, TAVG, TMAX, TMIN, PRCP, and 99.73% for PREC, at the right alignment. **0 unexplained mismatches** | Our request and `parse.snotel_to_long` reproduce NRCS's published values |
| Date semantics | With `periodRef=END`, SNWD/WTEQ/PREC dated *d* = the reading at the **end of day *d*** (the Report Generator labels it "start of *d*+1"). Daily aggregates are on the same date | Target "depth 30 days after *t*" = SNWD dated *t*+30. Weather on day *d* pairs with SNWD dated *d* |
| Residual mismatch 1 | PREC on Oct 1: water-year reset (Report Generator shows 0) | Cleaning: difference PREC **within** a water year, or use PRCP |
| Residual mismatch 2 | Extra AWDB values are all QC flag **S** (suspect), returned because of `returnSuspectData=true`, e.g. TMIN = 2148.6 °F | Cleaning must drop S values. The Report Generator hides them, so its data is pre-filtered |
| Open-Meteo daily vs hourly (Aspen, 2023–24, ERA5-Land and ERA5-seamless) | `snow_depth_mean/max` equal the **local-day** mean/max of the hourly values (100% of 182 days within rounding). UTC-day aggregation disagrees on 34–50% of days | `timezone=auto` daily values are local-calendar-day values. Always request `timezone=auto` |
| Open-Meteo temperature aggregation | Pending: Open-Meteo hourly rate limit at run time. Re-run §3b | — |
| Unit tests (`tests/test_lineage_parse.py`) | All pass, no network | Lineage, caching, manifest, offline mode, migration and parsers are tested |
| `verify_raw.py` | All cached raw files match their manifest SHA-256 | Raw data unchanged since download |

Caveat: SNOTEL reports in local *standard* time year-round, while Open-Meteo's `America/Denver` uses DST from mid-March.
Late-season days may be offset by 1 h, which is negligible for daily snow depth. Notebook 01 §3's lag test checks the alignment empirically.